In [1]:
import json
import os
import pandas as pd
import janitor  # noqa: F401

JAN_DIR   = "../data/blacklight_json"
APR_DIR   = "../data/blacklight_json_2026"
TARGETS   = "../data/rescan_targets_2026.csv"
YG_DOMAIN = "../data/yg/yg_ind_domain.csv"

### Tracker prevalence: Jan 2025 vs. Apr 2026

In [2]:
def indicators(path):
    with open(path) as f:
        d = json.load(f)
    groups = d.get('groups') or []
    if not groups:
        return {}
    return {c['cardType']: bool(c.get('testEventsFound'))
            for c in groups[0].get('cards', [])
            if c.get('cardType')}

In [3]:
targets = pd.read_csv(TARGETS).clean_names()

rows = []
for domain in targets['private_domain']:
    fname = domain.replace('.', '_') + '.json'
    apr_path = os.path.join(APR_DIR, fname)
    if not os.path.exists(apr_path):
        continue
    jan_path = os.path.join(JAN_DIR, fname)
    row = {'domain': domain}
    apr = indicators(apr_path)
    jan = indicators(jan_path) if os.path.exists(jan_path) else {}
    for ct in set(jan) | set(apr):
        row[f'jan_{ct}'] = jan.get(ct)
        row[f'apr_{ct}'] = apr.get(ct)
    rows.append(row)

df = pd.DataFrame(rows)
df.shape

(500, 19)

In [6]:
CATEGORIES = [
    ('ddg_join_ads',          'Ad trackers'),
    ('cookies',               'Third-party cookies'),
    ('canvas_fingerprinters', 'Canvas fingerprinting'),
    ('session_recorders',     'Session recording'),
    ('key_logging',           'Key logging'),
    ('fb_pixel_events',       'Facebook Pixel'),
    ('ga',                    'Google Analytics (remarketing)'),
    ('tiktok_pixel_events',   'TikTok Pixel'),
    ('twitter_pixel_events',  'X Pixel'),
]

out = []
for ct, label in CATEGORIES:
    jan = df.get(f'jan_{ct}')
    apr = df[f'apr_{ct}'].fillna(False)
    comparable = jan is not None and jan.notna().any()
    jan = jan.fillna(False) if comparable else None
    out.append({
        'category': label,
        'jan_prev_%': jan.mean() * 100 if comparable else None,
        'apr_prev_%': apr.mean() * 100,
        'delta_pp': (apr.mean() - jan.mean()) * 100 if comparable else None,
    })

stability

,category,jan_prev_%,apr_prev_%,delta_pp
0,Ad trackers,67.4,67.2,-0.2
1,Third-party cookies,57.6,53.6,-4.0
2,Canvas fingerprinting,12.4,20.6,8.2
3,Session recording,10.0,9.2,-0.8
4,Key logging,3.8,4.2,0.4
5,Facebook Pixel,21.6,16.6,-5.0
6,Google Analytics (remarketing),2.8,23.8,21.0
7,TikTok Pixel,NaN,3.8,NaN
8,X Pixel,NaN,6.6,NaN


### Coverage of the 500 rescanned domains

In [7]:
yg = pd.read_csv(YG_DOMAIN).clean_names()
target_domains = set(df['domain'])
sub = yg.query('private_domain in @target_domains')
sub.shape

(37875, 4)

In [8]:
coverage = pd.Series({
    'n_domains':            len(target_domains),
    'panelists_touched':    sub['caseid'].nunique(),
    'pct_panelists':        100 * sub['caseid'].nunique() / yg['caseid'].nunique(),
    'pct_total_visits':     100 * sub['visits'].sum() / yg['visits'].sum(),
    'pct_total_duration':   100 * sub['duration'].sum() / yg['duration'].sum(),
}).round(2)
coverage

n_domains              500.00
panelists_touched     1129.00
pct_panelists           99.56
pct_total_visits        58.91
pct_total_duration      59.32
dtype: float64

In [9]:
# Per-panelist share: how much of an individual's browsing happens on these 500 domains?# 
total  = yg.groupby('caseid')[['visits', 'duration']].sum()
in_tgt = sub.groupby('caseid')[['visits', 'duration']].sum()

shares = (in_tgt / total).rename(
    columns={'visits': 'visit_share', 'duration': 'duration_share'}
)
shares.describe().round(3)

,visit_share,duration_share
count,1129.000,1129.000
mean,0.543,0.542
std,0.196,0.240
min,0.002,0.000
25%,0.398,0.365
50%,0.559,0.566
75%,0.689,0.727
max,1.000,1.000
